# SEBAL ET — Landsat 8/9 + ERA5-Land (geeSEBAL patched สำหรับนอก CONUS)

- **Patch** `fexp_hot_pixel`: เปลี่ยน Tfac จาก GRIDMET (ใช้ได้แค่ CONUS) → **CHIRPS** (ฝน 50S–50N) + **ERA5-Land** (PET, global) ตาม Allen et al. 2013 Eqn 8
- **Forcing**: ERA5-Land hourly/daily แปลงชื่อ band + หน่วยให้ตรงกับที่ geeSEBAL ต้องการ
- พื้นที่ตัวอย่าง: โคราช (102.3–102.8E, 15.0–15.4N), ม.ค.–มี.ค. 2024


!pip install openet-geesebal earthengine-api geemap

In [1]:
# try:
#     from google.colab import output
#     output.enable_custom_widget_manager()
# except ImportError:
#     pass

import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-dancingriver2')


In [2]:
import openet.geesebal.model as gsmodel
from openet.geesebal.model import homogeneous_mask

# กัน re-run cell แล้ว _orig ชี้ไปที่ตัว patch เอง
if not hasattr(gsmodel, '_orig_fexp_hot_pixel'):
    gsmodel._orig_fexp_hot_pixel = gsmodel.fexp_hot_pixel


def fexp_hot_pixel_global(
        time_start, albedo, ndvi, ndwi, lst_dem, rn, g,
        ndvi_hot, lst_hot, geometry_image, coords, proj, dem,
        calibration_points=10,
):
    """เหมือน geeSEBAL ต้นฉบับ แต่เปลี่ยน Tfac จาก GRIDMET (CONUS)
       -> CHIRPS (ฝน, global 50S-50N) + ERA5-Land (PET, global)"""

    pos_ndvi   = ndvi.updateMask(ndvi.gte(0)).rename('post_ndvi')
    ndvi_neg   = pos_ndvi.multiply(-1).rename('ndvi_neg')
    lst_neg    = lst_dem.multiply(-1).rename('lst_neg')
    lst_nw     = lst_dem.updateMask(ndwi.lte(0)).rename('lst_nw')
    stdev_ndvi = homogeneous_mask(ndvi, proj)

    images = pos_ndvi.addBands([ndvi, ndvi_neg, rn, g, pos_ndvi, lst_neg, lst_nw, coords])

    # --- เปอร์เซ็นไทล์ NDVI ต่ำ (ดินเปล่า) ---
    d_ndvi = (images.select('post_ndvi').updateMask(stdev_ndvi)
              .reduceRegion(ee.Reducer.percentile([ndvi_hot]), geometry_image, 30, maxPixels=1e9)
              .combine(ee.Dictionary({'post_ndvi': 100}), overwrite=False))
    i_low_NDVI = images.updateMask(
        images.select('post_ndvi').lte(ee.Number(d_ndvi.get('post_ndvi'))))

    # --- เปอร์เซ็นไทล์ LST สูง ---
    d_lst = (i_low_NDVI.select('lst_neg').updateMask(stdev_ndvi)
             .reduceRegion(ee.Reducer.percentile([lst_hot]), geometry_image, 30, maxPixels=1e9)
             .combine(ee.Dictionary({'lst_neg': 350}), overwrite=False))
    i_top_LST = (i_low_NDVI.updateMask(stdev_ndvi)
                 .updateMask(i_low_NDVI.select('lst_neg').lte(ee.Number(d_lst.get('lst_neg')))))

    c_int        = i_top_LST.select('lst_nw').min(1).max(1).int().rename('int')
    c_lst_hotpix = i_top_LST.addBands(c_int)

    # ===== * Tfac แบบ global (Allen et al. 2013 Eqn 8) =====
    d0, d1 = ee.Date(time_start).advance(-60, 'days'), ee.Date(time_start)

    # ฝนสะสม 60 วัน (mm) — CHIRPS ครอบคลุม 50S-50N รวมไทย
    p60 = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
           .filterDate(d0, d1).select('precipitation').sum())

    # PET สะสม 60 วัน (mm) — ERA5-Land (หน่วย m, ค่าติดลบตามคอนเวนชัน ECMWF)
    pet60 = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
             .filterDate(d0, d1).select('potential_evaporation_sum').sum()
             .abs().multiply(1000).max(1))          # -> mm, กันหารศูนย์

    ratio = p60.divide(pet60)
    Tfac  = ee.Image(ratio.expression('2.6 - 13 * r', {'r': ratio})
                     .where(ratio.gt(0.2), 0)).rename('Tfac').unmask(0)   # unmask กัน dropNulls
    # ===================================================

    c_lst_hotpix = c_lst_hotpix.addBands(Tfac).select(
        ['ndvi', 'rn_inst', 'g_inst', 'lst_nw', 'longitude', 'latitude', 'int', 'Tfac'])

    n_hot = ee.Number(c_lst_hotpix.select('int')
                      .reduceRegion(ee.Reducer.sum(), geometry_image, 30, maxPixels=1e9)
                      .get('int'))

    return ee.FeatureCollection(ee.Algorithms.If(
        n_hot.gt(3000),
        c_lst_hotpix.stratifiedSample(
            numPoints=calibration_points, classBand='int',
            region=geometry_image, scale=30, dropNulls=True, geometries=True),
        ee.FeatureCollection([])
    ))


# เขียนทับฟังก์ชันเดิม (openet_image เรียกผ่าน model.et -> module globals จึงมีผลจริง)
gsmodel.fexp_hot_pixel = fexp_hot_pixel_global
print('✅ patched fexp_hot_pixel -> CHIRPS + ERA5-Land')


✅ patched fexp_hot_pixel -> CHIRPS + ERA5-Land


## เตรียม forcing จาก ERA5-Land

หน่วยที่ geeSEBAL ต้องการ (inst): `temperature` **°C**, `specific_humidity` kg/kg,
`pressure` Pa, `wind_u/wind_v` m/s, `shortwave_radiation` W/m² —
ส่วน daily: `srad` W/m², `tmmn`/`tmmx` **คงเป็น K** (โมเดล subtract 273.15 เอง)


In [3]:
import openet.geesebal as geesebal

aoi = ee.Geometry.Rectangle([102.3, 15.0, 102.8, 15.4])
START, END = '2024-01-01', '2024-03-31'

def era5_inst(img):
    t  = img.select('temperature_2m').subtract(273.15).rename('temperature')   # K -> degC
    p  = img.select('surface_pressure').rename('pressure')                     # Pa
    u  = img.select('u_component_of_wind_10m').rename('wind_u')
    v  = img.select('v_component_of_wind_10m').rename('wind_v')
    sw = (img.select('surface_solar_radiation_downwards_hourly')
             .divide(3600).rename('shortwave_radiation'))                      # J/m2 -> W/m2

    td = img.select('dewpoint_temperature_2m').subtract(273.15)                # degC
    ea = td.expression('0.6108*exp((17.27*Td)/(Td+237.3))', {'Td': td})        # kPa
    # โมเดลคำนวณย้อนกลับ: ea = (1/0.622)*q*P_kPa  -> q = 0.622*ea/P_kPa
    q  = ea.multiply(0.622).divide(p.divide(1000)).rename('specific_humidity')

    return ee.Image.cat([t, q, p, u, v, sw]).copyProperties(img, ['system:time_start'])

# daily: tmmn/tmmx เป็น K (โมเดล subtract 273.15 ให้เอง) — ห้ามแปลง!
def era5_daily(img):
    srad = img.select('surface_solar_radiation_downwards_sum').divide(86400).rename('srad')  # W/m2
    tmmn = img.select('temperature_2m_min').rename('tmmn')   # K
    tmmx = img.select('temperature_2m_max').rename('tmmx')   # K
    return ee.Image.cat([srad, tmmn, tmmx]).copyProperties(img, ['system:time_start'])

METEO_INST  = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate(START, END).map(era5_inst)
METEO_DAILY = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filterDate(START, END).map(era5_daily)


In [4]:
coll = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
        .merge(ee.ImageCollection('LANDSAT/LC08/C02/T1_L2'))
        .filterBounds(aoi).filterDate(START, END)
        .filter(ee.Filter.lt('CLOUD_COVER', 10))
        .sort('CLOUD_COVER'))

n_scene = coll.size().getInfo()
assert n_scene > 0, '❌ ไม่พบภาพ Landsat (cloud < 10%) ในช่วงวันที่กำหนด — ลองขยาย START/END หรือเพิ่ม CLOUD_COVER'

ls_img   = ee.Image(coll.first())
scene_id = ls_img.get('system:index').getInfo()
date_str = ee.Date(ls_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
print(f'พบ {n_scene} ภาพ | เลือก: {scene_id}  ({date_str})')


พบ 9 ภาพ | เลือก: 2_LC08_128049_20240207  (2024-02-07)


In [ ]:
# ===== PRE-FLIGHT CHECK — ตรวจ forcing ก่อนรัน =====
t0 = ee.Date(ls_img.get('system:time_start'))
mi = ee.Image(METEO_INST.filterDate(t0.advance(-2, 'hour'), t0.advance(2, 'hour')).first())
md = ee.Image(METEO_DAILY.filterDate(t0.advance(-1, 'day'), t0).first())

fi = mi.reduceRegion(ee.Reducer.mean(), aoi, 5000, maxPixels=1e9).getInfo()
fd = md.reduceRegion(ee.Reducer.mean(), aoi, 5000, maxPixels=1e9).getInfo()

def show(d):
    for k, v in sorted(d.items()):
        print(f'  {k:22s} ' + (f'{v:.4f}' if v is not None else 'None ❌'))

print('--- forcing (inst) ---');  show(fi)
print('--- forcing (daily) ---'); show(fd)

# ค่าที่ควรได้แถบโคราช ม.ค.-มี.ค. เวลา ~10:30 น. ท้องถิ่น:
#   temperature         ~ 26-33      (degC  <- ถ้าเห็น ~300 แปลว่ายังเป็น K อยู่!)
#   specific_humidity   ~ 0.010-0.018
#   pressure            ~ 99000-101000
#   shortwave_radiation ~ 600-900
#   srad                ~ 200-260    (W/m2 เฉลี่ยทั้งวัน)
#   tmmn / tmmx         ~ 288-305    (K)
assert fi['temperature'] is not None and 5 < fi['temperature'] < 45, '❌ temperature ไม่ใช่ degC!'
assert fd['tmmx'] is not None and fd['tmmx'] > 200, '❌ tmmx ควรเป็น K (อย่าแปลงหน่วย daily)!'
print('✅ forcing ผ่าน')


--- forcing (inst) ---
  pressure               99462.9284
  shortwave_radiation    365.5851
  specific_humidity      0.0130
  temperature            28.8167
  wind_u                 2.0817
  wind_v                 0.3488
--- forcing (daily) ---
  srad                   252.5286
  tmmn                   295.3504
  tmmx                   309.3011
✅ forcing ผ่าน


In [5]:
model = geesebal.Image.from_landsat_c2_sr(
    ls_img,
    meteorology_source_inst  = METEO_INST,
    meteorology_source_daily = METEO_DAILY,
    elev_source = 'USGS/SRTMGL1_003',
    ndvi_cold=5, lst_cold=20, ndvi_hot=10, lst_hot=20,
    calibration_points=6, max_iterations=15,
)

et     = model.et.clip(aoi)
ndvi   = model.ndvi.clip(aoi)
lst_c  = model.lst.subtract(273.15).clip(aoi)
albedo = model.albedo.clip(aoi)

# reducer แบบ combine จะได้ key เป็น et_mean / et_min / et_max (ไม่ใช่ 'et')
s = et.reduceRegion(
    ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
    aoi, 60, maxPixels=1e9, bestEffort=True).getInfo()

print(f"ET (mm/day): mean={s.get('et_mean')}, min={s.get('et_min')}, max={s.get('et_max')}")
assert s.get('et_mean') is not None, '⚠️ ET ว่าง — ลองผ่อน ndvi_hot=20, calibration_points=3'


ET (mm/day): mean=1.8112511152475634, min=0, max=5.152297127551847


In [6]:
vis_et   = {'min': 0,  'max': 8,   'palette': ['#a50026', '#f46d43', '#fee090', '#e0f3f8', '#74add1', '#313695']}
vis_ndvi = {'min': 0,  'max': 0.9, 'palette': ['#8c510a', '#d8b365', '#f6e8c3', '#c7eae5', '#5ab4ac', '#01665e']}
vis_lst  = {'min': 22, 'max': 45,  'palette': ['#2166ac', '#67a9cf', '#f7f7f7', '#fddbc7', '#ef8a62', '#b2182b']}

M = geemap.Map(center=[15.2, 102.55], zoom=11, basemap='HYBRID')
M.addLayer(ls_img.clip(aoi), {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 7000, 'max': 18000}, 'Landsat RGB', False)
M.addLayer(albedo, {'min': 0.05, 'max': 0.35}, 'Albedo', False)
M.addLayer(ndvi,   vis_ndvi, 'NDVI', False)
M.addLayer(lst_c,  vis_lst,  'LST (°C)', False)
M.addLayer(et,     vis_et,   f'SEBAL ET {date_str} (mm/day)', True)
M.addLayer(ee.Image().paint(aoi, 0, 2), {'palette': 'yellow'}, 'AOI')
M.add_colorbar(vis_et, label='SEBAL ET (mm/day)', orientation='horizontal')
M.addLayerControl()
M


Map(center=[15.2, 102.55], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…